# Autocomplete and Autocorrect Data Analytics
## 3. Autocorrect Implementation and Evaluation

This notebook covers:
- Implementing autocorrect algorithms
- Training models on text data
- Creating synthetic spelling error test sets
- Evaluating performance metrics

## 1. Setup and Load Data

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import time

# Import custom modules
from autocorrect import (
    SimpleAutocorrect,
    EditDistanceAutocorrect,
    ContextawareAutocorrect,
    AutocorrectEvaluator
)
from utils import VisualizationHelper

VisualizationHelper.set_style()
%matplotlib inline

print("Libraries loaded successfully!")

In [ ]:
# Load vocabulary
with open('../data/vocabulary.txt', 'r') as f:
    vocabulary = [line.strip() for line in f.readlines()]

print(f"Vocabulary size: {len(vocabulary)}")
print(f"Sample words: {vocabulary[:20]}")

## 2. Create Synthetic Misspelling Test Set

In [ ]:
def generate_misspellings(word, num_variations=3):
    """Generate common misspellings of a word"""
    misspellings = []
    
    # Delete character
    for i in range(len(word)):
        misspelled = word[:i] + word[i+1:]
        if misspelled and len(misspellings) < num_variations:
            misspellings.append(misspelled)
    
    # Replace character
    for i in range(len(word)):
        for char in 'abcdefghijklmnopqrstuvwxyz':
            if char != word[i]:
                misspelled = word[:i] + char + word[i+1:]
                if len(misspellings) < num_variations:
                    misspellings.append(misspelled)
                else:
                    break
        if len(misspellings) >= num_variations:
            break
    
    return misspellings[:num_variations]

# Create test set
test_vocabulary = vocabulary[:20]  # Use first 20 words
test_cases = []

for word in test_vocabulary:
    misspellings = generate_misspellings(word, num_variations=2)
    for misspelled in misspellings:
        test_cases.append((misspelled, word))

print(f"Created {len(test_cases)} test cases")
print("\nFirst 10 test cases:")
for i, (misspelled, correct) in enumerate(test_cases[:10]):
    print(f"  {i+1}. '{misspelled}' -> '{correct}'")

## 3. Build Autocorrect Algorithms

### 3.1 Edit Distance Autocorrect

In [ ]:
# Train Edit Distance autocorrect
print("Training Edit Distance autocorrect...")
start_time = time.time()

edit_distance_corrector = EditDistanceAutocorrect(vocabulary=vocabulary, max_distance=2)
edit_distance_corrector.train(vocabulary)

ed_train_time = time.time() - start_time
print(f"Training time: {ed_train_time:.4f}s")

# Test corrections
test_word = test_cases[0][0]
corrections = edit_distance_corrector.correct_word(test_word, top_k=5)
print(f"\nCorrections for '{test_word}': {corrections}")

### 3.2 Context-aware Autocorrect

In [ ]:
# Train Context-aware autocorrect
print("Training Context-aware autocorrect...")
start_time = time.time()

context_corrector = ContextawareAutocorrect(vocabulary=vocabulary)
context_corrector.train(vocabulary, context_window=2)

ctx_train_time = time.time() - start_time
print(f"Training time: {ctx_train_time:.4f}s")

# Test corrections
test_word = test_cases[0][0]
correction = context_corrector.correct_with_context(test_word, max_distance=2)
print(f"\nCorrection for '{test_word}': {correction}")

## 4. Performance Evaluation

In [ ]:
# Evaluate models
models = {
    'EditDistance': edit_distance_corrector,
    'ContextAware': context_corrector
}

results = {}

for model_name, model in models.items():
    print(f"\nEvaluating {model_name}...")
    
    accuracies = []
    wers = []
    cers = []
    query_times = []
    
    for misspelled, correct in test_cases:
        # Time the correction
        start = time.time()
        if hasattr(model, 'correct_word'):
            corrected = model.correct_word(misspelled, top_k=1)[0][0] if model.correct_word(misspelled, top_k=1) else misspelled
        else:
            corrected = model.correct_with_context(misspelled)
        query_time = time.time() - start
        
        # Calculate metrics
        accuracy = 1.0 if corrected == correct else 0.0
        wer = AutocorrectEvaluator.word_error_rate(misspelled, corrected)
        cer = AutocorrectEvaluator.character_error_rate(misspelled, corrected)
        
        accuracies.append(accuracy)
        wers.append(wer)
        cers.append(cer)
        query_times.append(query_time)
    
    results[model_name] = {
        'accuracy': np.mean(accuracies),
        'avg_wer': np.mean(wers),
        'avg_cer': np.mean(cers),
        'avg_query_time': np.mean(query_times),
        'training_time': ed_train_time if model_name == 'EditDistance' else ctx_train_time
    }
    
    print(f"  Accuracy: {results[model_name]['accuracy']:.4f}")
    print(f"  Avg WER: {results[model_name]['avg_wer']:.4f}")
    print(f"  Avg CER: {results[model_name]['avg_cer']:.4f}")
    print(f"  Avg Query Time: {results[model_name]['avg_query_time']*1000:.4f}ms")

## 5. Results Summary

In [ ]:
# Create results dataframe
results_df = pd.DataFrame(results).T
print("\nPerformance Comparison:")
print(results_df)

# Save results
results_df.to_csv('../results/autocorrect_comparison.csv')
print("\n✓ Results saved to '../results/autocorrect_comparison.csv'")

In [ ]:
# Visualize performance comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Accuracy Comparison
accuracy_values = [results[model]['accuracy'] for model in models.keys()]
axes[0, 0].bar(models.keys(), accuracy_values, color=['steelblue', 'darkgreen'])
axes[0, 0].set_ylabel('Score')
axes[0, 0].set_title('Accuracy')
axes[0, 0].set_ylim(0, 1)

# WER Comparison
wer_values = [results[model]['avg_wer'] for model in models.keys()]
axes[0, 1].bar(models.keys(), wer_values, color=['steelblue', 'darkgreen'])
axes[0, 1].set_ylabel('Error Rate')
axes[0, 1].set_title('Average Word Error Rate')

# CER Comparison
cer_values = [results[model]['avg_cer'] for model in models.keys()]
axes[1, 0].bar(models.keys(), cer_values, color=['steelblue', 'darkgreen'])
axes[1, 0].set_ylabel('Error Rate')
axes[1, 0].set_title('Average Character Error Rate')

# Query Time Comparison
time_values = [results[model]['avg_query_time']*1000 for model in models.keys()]
axes[1, 1].bar(models.keys(), time_values, color=['steelblue', 'darkgreen'])
axes[1, 1].set_ylabel('Time (ms)')
axes[1, 1].set_title('Average Query Time')

plt.tight_layout()
plt.savefig('../results/autocorrect_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Figure saved to '../results/autocorrect_comparison.png'")